In [3]:
import os
import numpy as np
import torch
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import temp  as worker
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes
import cv2


In [4]:
if __name__ == "__main__":
    worker.main()

here
here1
here2
here3
here4


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Users/danhuang/miniconda3/envs/enertiv/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 302, in _worker_loop
    data = fetcher.fetch(index)
  File "/Users/danhuang/miniconda3/envs/enertiv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 58, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/Users/danhuang/miniconda3/envs/enertiv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 58, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/Users/danhuang/miniconda3/envs/enertiv/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 295, in __getitem__
    return self.dataset[self.indices[idx]]
  File "/Users/danhuang/Desktop/CUB/spring_2023/CSCI4318/enertiv-meter-reading-api/prototype/temp.py", line 51, in __getitem__
    pos = np.where(masks[i])

In [2]:
# ig = Image.open('PennFudanPed/PNGImages/FudanPed00001.png').convert("RGB")
# img,_ = worker.get_transform(train=True)(ig, None)
# img_int = torch.tensor(img*255, dtype=torch.uint8)
# model = worker.load_model()
# model.eval()


/var/folders/zk/x1_f408d12jcxwxdxj_61yjh0000gn/T/ipykernel_2535/4115387233.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  img_int = torch.tensor(img*255, dtype=torch.uint8)


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [3]:
with torch.no_grad():
    pred = model([img])

bboxes, labels, scores = pred[0]['boxes'], pred[0]['labels'], pred[0]['scores']
num = torch.argwhere(scores > 0.5).shape[0]
igg = cv2.imread('PennFudanPed/PNGImages/FudanPed00001.png')
for i in range(num):
    x1, y1, x2, y2 = bboxes[5].numpy().astype(int)
   
    igg = cv2.rectangle(igg, (x1, y1), (x2, y2), (255, 0, 0), 2)
fig = plt.figure(figsize=(14, 10))
plt.imshow(igg)

AttributeError: 'AxesImage' object has no attribute 'permute'

In [74]:
import os
import numpy as np
import torch
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from functools import lru_cache

import sys
    # caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, 'detection/')

from engine import train_one_epoch, evaluate
import utils
import transforms as T

class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms):
        self.root = root
        self.transforms = transforms
        # load all image files, sorting them to
        # ensure that they are aligned
        self.imgs = list(sorted(os.listdir(os.path.join(root, "JPEGImages"))))
        self.masks = list(sorted(os.listdir(os.path.join(root, "SegmentationClass"))))
        # self.imgs = list(sorted(os.listdir(os.path.join(root, "PNGImages"))))
        # self.masks = list(sorted(os.listdir(os.path.join(root, "PedMasks"))))

    def __getitem__(self, idx):
        # load images and masks
        img_path = os.path.join(self.root, "JPEGImages", self.imgs[idx])
        mask_path = os.path.join(self.root, "SegmentationClass", self.masks[idx])
        # img_path = os.path.join(self.root, "PNGImages", self.imgs[idx])
        # mask_path = os.path.join(self.root, "PedMasks", self.masks[idx])
        img = Image.open(img_path).convert("RGB")
        # note that we haven't converted the mask to RGB,
        # because each color corresponds to a different instance
        # with 0 being background
        mask = Image.open(mask_path)
        # convert the PIL Image into a numpy array
        mask = np.array(mask)
        print(mask.shape)
        # instances are encoded as different colors
        obj_ids = np.unique(mask)
        print(obj_ids)
        # first id is the background, so remove it
        obj_ids = obj_ids[1:]

        # split the color-encoded mask into a set
        # of binary masks
        masks = mask == obj_ids[:, None, None]
        print(masks)
        # get bounding box coordinates for each mask
        num_objs = len(obj_ids)
        boxes = []
        for i in range(num_objs):
            pos = np.where(masks[i])
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            boxes.append([xmin, ymin, xmax, ymax])

        # convert everything into a torch.Tensor
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        # there is only one class
        labels = torch.ones((num_objs,), dtype=torch.int64)
        masks = torch.as_tensor(masks, dtype=torch.uint8)

        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        # suppose all instances are not crowd
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["masks"] = masks
        target["image_id"] = image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)

def get_transform(train):
    transforms = []
    transforms.append(T.PILToTensor())
    transforms.append(T.ConvertImageDtype(torch.float))
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    return T.Compose(transforms)

In [77]:
dataset = PennFudanDataset('Rough-Digit-Classification', get_transform(train=True))
#dataset = PennFudanDataset('PennFudanPed', get_transform(train=True))
dataset.__getitem__(0)
# (426, 385)
# (3,)
# (2, 426, 385)

(480, 640, 3)
[  0  64 128]
False


/var/folders/zk/x1_f408d12jcxwxdxj_61yjh0000gn/T/ipykernel_2875/399141682.py:51: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  masks = mask == obj_ids


TypeError: 'bool' object is not subscriptable